# FIONA Treatment Example

This notebook tests `shrecc.tyndp` and `shrecc.premise_mapping` from the hourly TYNDP pickle files. It builds `Z_gross`, computes the hourly consumption mix, and maps it to premise regions and activities.

In [21]:
import sys
from pathlib import Path

import numpy as np
import pandas as pd
import xarray as xr

# The following helps to import modules not from a parent folder
cwd = Path.cwd().resolve()

for candidate in [cwd, *cwd.parents]:
    if (candidate / "shrecc" / "tyndp.py").exists():
        project_root = candidate
        break
    if (candidate / "shrecc" / "shrecc" / "tyndp.py").exists():
        project_root = candidate / "shrecc"
        break
else:
    raise RuntimeError("Could not find the local shrecc package root")

if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))

from shrecc.premise_mapping import (
    PremiseConsumptionMixMapper,
    build_premise_region_map,
    map_consumption_mix_regions_xr,
    map_consumption_mix_technologies_xr,
)
from shrecc.tyndp import (
    build_z_gross_from_tyndp_excel,
    build_z_gross_from_tyndp_pickles,
    consumption_mix_from_tyndp_excel,
    consumption_mix_from_tyndp_pickles,
    consumption_mix_from_z_gross,
)


In [22]:
data_dir = project_root / "data" / "tyndp"

tyndp_workbook = data_dir / "MMStandardOutputFile_DE2050_Plexos_CY2009_v11_SoS.xlsb"
production_pickle = data_dir / "DE2050_CY2009_prod.pkl"
trade_pickle = data_dir / "DE2050_CY2009_trade.pkl"
technology_mapping = data_dir / "tyndp_ei_mapping.xlsx"
country_mapping = data_dir / "tyndp_country_mapping.xlsx"

for path in [tyndp_workbook, production_pickle, trade_pickle, technology_mapping, country_mapping]:
    assert path.exists(), path

## Build `Z_gross`

`Z_gross` is the gross hourly supply table with production and trade flows in one column structure.

In [23]:
Z_gross = build_z_gross_from_tyndp_pickles(
    production_pickle=production_pickle,
    trade_pickle=trade_pickle,
    technology_mapping=technology_mapping,
    country_mapping=country_mapping,
    verbose=True,
)

Z_gross.shape, Z_gross.columns.names, Z_gross.index[:3]

2026-05-12 10:44:14 Loading TYNDP pickle files
2026-05-12 10:44:14 Parsing hourly datetime index
2026-05-12 10:44:14 Loading technology and country mappings
2026-05-12 10:44:15 Aggregating hourly production by country and technology


D:\shrecc_project\shrecc\shrecc\treatment_fiona.py:853: FutureWarning: DataFrame.groupby with axis=1 is deprecated. Do `frame.T.groupby(...)` without axis instead.
  production = production.groupby(


2026-05-12 10:44:15 Aggregating hourly trade to positive directional country flows


D:\shrecc_project\shrecc\shrecc\treatment_fiona.py:298: UserWarning: Dropping trade columns without country-pair mappings: AT00-AT00EV, AT00-AT00RETE, AT00EV-AT00, AT00EV-AT00RETE, AT00RETE-AT00EV, BE00-BE00EV, BE00-BE00RETE, BE00EV-BE00, BE00EV-BE00RETE, BE00RETE-BE00EV
  trade_directional = _aggregate_tyndp_trade(trade, connections)
D:\shrecc_project\shrecc\shrecc\treatment_fiona.py:915: FutureWarning: DataFrame.groupby with axis=1 is deprecated. Do `frame.T.groupby(...)` without axis instead.
  trade = trade.groupby(level=["Country from", "Country to"], axis=1).sum()


2026-05-12 10:44:15 Combining production and trade into Z_gross
2026-05-12 10:44:15 Built Z_gross with shape (8736, 485)


((8736, 485),
 FrozenList(['type', 'country from', 'country to', 'source']),
 DatetimeIndex(['2050-01-01 00:00:00', '2050-01-01 01:00:00',
                '2050-01-01 02:00:00'],
               dtype='datetime64[ns]', freq=None))

In [24]:
assert len(Z_gross) == 8736
assert Z_gross.columns.names == ["type", "country from", "country to", "source"]
assert {"production mix", "trade"}.issubset(Z_gross.columns.get_level_values("type"))

Z_gross.head(2)

type                   production mix                                     \
country from                       AL                                      
country to                         AL                                      
source              Gas CCGT new [MW] Gas CCGT old 2 [MW] Reservoir [MW]   
2050-01-01 00:00:00               110                   0           1904   
2050-01-01 01:00:00               100                   0           1904   

type                                                             \
country from                                                      
country to                                                        
source              Run-of-River [MW] Solar (Photovoltaic) [MW]   
2050-01-01 00:00:00               193                         0   
2050-01-01 01:00:00               193                         0   

type                                                      \
country from                                          AT   
country to                                            AT   
source              Wind Onshore [MW] Hydrogen CCGT [MW]   
2050-01-01 00:00:00                40               2934   
2050-01-01 01:00:00                43               2934   

type                                                                 \
country from                                                          
country to                                                            
source              Others non-renewable [MW] Others renewable [MW]   
2050-01-01 00:00:00                       464                   502   
2050-01-01 01:00:00                       464                   502   

type                                                           ... trade  \
country from                                                   ...    SI   
country to                                                     ...    IT   
source              Pump Storage - Closed Loop (turbine) [MW]  ... trade   
2050-01-01 00:00:00                                        31  ...     0   
2050-01-01 01:00:00                                        31  ...     0   

type                                                                       
country from           SK                UK                                
country to             CZ    HU    PL    BE    DE    DK    FR    IE    NL  
source              trade trade trade trade trade trade trade trade trade  
2050-01-01 00:00:00  2600     0     0  1000  2800     0     0     0  1000  
2050-01-01 01:00:00  2600     0     0  1000  2800     0     0     0  1000  

[2 rows x 485 columns]

## Compute Consumption Mix From `Z_gross`

The debug output includes the direct production/trade coefficient matrices used for the conservation checks.

In [25]:
consumption_mix_xr, debug = consumption_mix_from_z_gross(
    Z_gross,
    check=True,
    return_debug=True,
    verbose=True,
)

consumption_mix_xr

2026-05-12 10:44:18 Building domestic production and trade denominator matrices


D:\shrecc_project\shrecc\shrecc\treatment_fiona.py:1022: FutureWarning: DataFrame.groupby with axis=1 is deprecated. Do `frame.T.groupby(...)` without axis instead.
  .groupby(["country from", "country to"], axis=1)
D:\shrecc_project\shrecc\shrecc\treatment_fiona.py:1042: FutureWarning: DataFrame.groupby with axis=1 is deprecated. Do `frame.T.groupby(...)` without axis instead.
  country_total = Z_trade.groupby("country to", axis=1).sum()


2026-05-12 10:44:18 Calculating direct production and trade shares
2026-05-12 10:44:18 Using 35 countries and 32 technologies
2026-05-12 10:44:18 Reshaping production shares
2026-05-12 10:44:19 Reshaping trade shares
2026-05-12 10:44:20 Checking direct shares sum to one
2026-05-12 10:44:20 Solving hourly trade system
2026-05-12 10:44:27 Checking final consumption mix sums to one
2026-05-12 10:44:27 Building xarray DataArray
2026-05-12 10:44:49 Built consumption_mix_xr with shape (8736, 35, 35, 32)


<xarray.DataArray 'consumption_mix' (time: 8736, consumer_country: 35,
                                     source_country: 35, technology: 32)> Size: 3GB
array([[[[0.00000000e+00, 2.41652021e-02, 0.00000000e+00, ...,
          0.00000000e+00, 0.00000000e+00, 8.78734622e-03],
         [0.00000000e+00, 0.00000000e+00, 0.00000000e+00, ...,
          0.00000000e+00, 0.00000000e+00, 0.00000000e+00],
         [0.00000000e+00, 0.00000000e+00, 0.00000000e+00, ...,
          0.00000000e+00, 0.00000000e+00, 0.00000000e+00],
         ...,
         [0.00000000e+00, 0.00000000e+00, 0.00000000e+00, ...,
          0.00000000e+00, 0.00000000e+00, 0.00000000e+00],
         [0.00000000e+00, 0.00000000e+00, 0.00000000e+00, ...,
          0.00000000e+00, 0.00000000e+00, 0.00000000e+00],
         [0.00000000e+00, 0.00000000e+00, 0.00000000e+00, ...,
          0.00000000e+00, 0.00000000e+00, 0.00000000e+00]],

        [[0.00000000e+00, 1.61824034e-04, 0.00000000e+00, ...,
          0.00000000e+00, 0.00000000e+00, 5.88451033e-05],
         [0.00000000e+00, 0.00000000e+00, 0.00000000e+00, ...,
          0.00000000e+00, 0.00000000e+00, 1.41976107e-01],
         [0.00000000e+00, 0.00000000e+00, 0.00000000e+00, ...,
          0.00000000e+00, 0.00000000e+00, 1.57459439e-03],
...
          0.00000000e+00, 0.00000000e+00, 0.00000000e+00],
         [0.00000000e+00, 0.00000000e+00, 0.00000000e+00, ...,
          0.00000000e+00, 0.00000000e+00, 1.79452669e-02],
         [0.00000000e+00, 0.00000000e+00, 0.00000000e+00, ...,
          0.00000000e+00, 1.99220275e-04, 1.38429298e-04]],

        [[0.00000000e+00, 0.00000000e+00, 0.00000000e+00, ...,
          0.00000000e+00, 0.00000000e+00, 0.00000000e+00],
         [0.00000000e+00, 0.00000000e+00, 0.00000000e+00, ...,
          0.00000000e+00, 0.00000000e+00, 0.00000000e+00],
         [0.00000000e+00, 0.00000000e+00, 0.00000000e+00, ...,
          0.00000000e+00, 0.00000000e+00, 0.00000000e+00],
         ...,
         [0.00000000e+00, 0.00000000e+00, 0.00000000e+00, ...,
          0.00000000e+00, 0.00000000e+00, 0.00000000e+00],
         [0.00000000e+00, 0.00000000e+00, 0.00000000e+00, ...,
          0.00000000e+00, 0.00000000e+00, 0.00000000e+00],
         [0.00000000e+00, 0.00000000e+00, 0.00000000e+00, ...,
          0.00000000e+00, 5.15654365e-01, 3.58305258e-01]]]],
      shape=(8736, 35, 35, 32))
Coordinates:
  * time              (time) datetime64[ns] 70kB 2050-01-01 ... 2050-12-30T23...
  * consumer_country  (consumer_country) object 280B 'AL' 'AT' ... 'SK' 'UK'
  * source_country    (source_country) object 280B 'AL' 'AT' 'BA' ... 'SK' 'UK'
  * technology        (technology) object 256B 'Gas CCGT CCS [MW]' ... 'Wind ...

In [26]:
# Consumption mix in Luxembourg first hour of 2050
consumption_mix_xr.sel(consumer_country="LU", time='2050-01-01T00:00:00.000000000').to_dataframe()["consumption_mix"].nlargest(30)

source_country  technology               
BE              Gas CCGT new [MW]            0.161954
FR              Nuclear [MW]                 0.130657
                Wind Onshore [MW]            0.068856
LU              Others renewable [MW]        0.057029
FR              Reservoir [MW]               0.055759
                Wind Offshore [MW]           0.053465
DE              Hydrogen CCGT [MW]           0.050567
BE              Others non-renewable [MW]    0.038745
                Gas CCGT present 2 [MW]      0.038315
DE              Wind Onshore [MW]            0.037433
FR              Run-of-River [MW]            0.024439
ES              Wind Onshore [MW]            0.018983
FR              Gas CCGT present 2 [MW]      0.016285
BE              Others renewable [MW]        0.014953
UK              Others non-renewable [MW]    0.012853
LU              Wind Onshore [MW]            0.012506
UK              Wind Offshore [MW]           0.010811
LU              Run-of-River [MW]       

In [27]:
direct_total = debug["direct_total"]

direct_total.min(), direct_total.max()

(np.float64(0.9999999999999997), np.float64(1.0000000000000002))

In [28]:
mix_total = consumption_mix_xr.sum(["source_country", "technology"])

float(mix_total.min()), float(mix_total.max())

(0.9999999999999997, 1.0000000000000004)

In [29]:
np.testing.assert_allclose(direct_total, 1, atol=1e-8)
np.testing.assert_allclose(mix_total, 1, atol=1e-8)

print("All conservation checks passed.")

All conservation checks passed.


## One-Step Pipeline

The convenience wrapper should give the same result as building `Z_gross` explicitly first.

In [30]:
consumption_mix_direct = consumption_mix_from_tyndp_pickles(
    production_pickle=production_pickle,
    trade_pickle=trade_pickle,
    technology_mapping=technology_mapping,
    country_mapping=country_mapping,
    check=True,
    verbose=True,
)

xr.testing.assert_allclose(consumption_mix_direct, consumption_mix_xr)

consumption_mix_direct.shape

2026-05-12 10:45:06 Loading TYNDP pickle files
2026-05-12 10:45:06 Parsing hourly datetime index
2026-05-12 10:45:07 Loading technology and country mappings
2026-05-12 10:45:07 Aggregating hourly production by country and technology


D:\shrecc_project\shrecc\shrecc\treatment_fiona.py:853: FutureWarning: DataFrame.groupby with axis=1 is deprecated. Do `frame.T.groupby(...)` without axis instead.
  production = production.groupby(


2026-05-12 10:45:07 Aggregating hourly trade to positive directional country flows


D:\shrecc_project\shrecc\shrecc\treatment_fiona.py:298: UserWarning: Dropping trade columns without country-pair mappings: AT00-AT00EV, AT00-AT00RETE, AT00EV-AT00, AT00EV-AT00RETE, AT00RETE-AT00EV, BE00-BE00EV, BE00-BE00RETE, BE00EV-BE00, BE00EV-BE00RETE, BE00RETE-BE00EV
  trade_directional = _aggregate_tyndp_trade(trade, connections)
D:\shrecc_project\shrecc\shrecc\treatment_fiona.py:915: FutureWarning: DataFrame.groupby with axis=1 is deprecated. Do `frame.T.groupby(...)` without axis instead.
  trade = trade.groupby(level=["Country from", "Country to"], axis=1).sum()


2026-05-12 10:45:07 Combining production and trade into Z_gross
2026-05-12 10:45:07 Built Z_gross with shape (8736, 485)
2026-05-12 10:45:07 Building domestic production and trade denominator matrices


D:\shrecc_project\shrecc\shrecc\treatment_fiona.py:1022: FutureWarning: DataFrame.groupby with axis=1 is deprecated. Do `frame.T.groupby(...)` without axis instead.
  .groupby(["country from", "country to"], axis=1)
D:\shrecc_project\shrecc\shrecc\treatment_fiona.py:1042: FutureWarning: DataFrame.groupby with axis=1 is deprecated. Do `frame.T.groupby(...)` without axis instead.
  country_total = Z_trade.groupby("country to", axis=1).sum()


2026-05-12 10:45:08 Calculating direct production and trade shares
2026-05-12 10:45:08 Using 35 countries and 32 technologies
2026-05-12 10:45:08 Reshaping production shares
2026-05-12 10:45:09 Reshaping trade shares
2026-05-12 10:45:09 Checking direct shares sum to one
2026-05-12 10:45:09 Solving hourly trade system
2026-05-12 10:45:14 Checking final consumption mix sums to one
2026-05-12 10:45:15 Building xarray DataArray
2026-05-12 10:45:27 Built consumption_mix_xr with shape (8736, 35, 35, 32)


(8736, 35, 35, 32)

## Excel/XLSB Input

If the pickle files are not available, the same `Z_gross` table can be built directly from the original TYNDP workbook. Reading the `.xlsb` file is much slower than loading the pickle files, so this section is mainly a from-scratch check.

In [31]:
# Z_gross_from_excel = build_z_gross_from_tyndp_excel(
#     excel_file=tyndp_workbook,
#     technology_mapping=technology_mapping,
#     country_mapping=country_mapping,
#     # Optionally cache the slow Excel extraction for future runs:
#     # production_pickle=data_dir / "DE2050_CY2009_prod.pkl",
#     # trade_pickle=data_dir / "DE2050_CY2009_trade.pkl",
#     verbose=True,
# )

# Z_gross_from_excel.shape, Z_gross_from_excel.columns.names, Z_gross_from_excel.index[:3]

In [32]:
# pd.testing.assert_frame_equal(Z_gross_from_excel, Z_gross)

# print("Excel-built Z_gross matches pickle-built Z_gross.")

The one-step Excel wrapper reads the workbook, builds `Z_gross`, and computes `consumption_mix_xr` in one call. This repeats the long workbook read and matrix solve, so run it when you specifically want to test the full Excel path.

In [33]:
# consumption_mix_from_excel = consumption_mix_from_tyndp_excel(
#     excel_file=tyndp_workbook,
#     technology_mapping=technology_mapping,
#     country_mapping=country_mapping,
#     check=True,
#     verbose=True,
# )

# xr.testing.assert_allclose(consumption_mix_from_excel, consumption_mix_xr)

# consumption_mix_from_excel.shape

## Premise Geography Map

Map the FIONA source countries to the IAM regions used by premise. In the installed premise version, `remind` and `image` are available by default. `remind-eu` is included here to make availability explicit; if the topology is not installed, the column is filled with missing values and a warning is emitted.

In [50]:
source_countries = consumption_mix_xr.source_country.to_index()
remind_eu_topology = data_dir / "remind-eu-topology.json"

remind_eu_topology, remind_eu_topology.exists()
topology_files = {}
if remind_eu_topology.exists():
    topology_files["remind-eu"] = remind_eu_topology

premise_region_map = build_premise_region_map(
    source_countries,
    iam_models=("remind", "image", "remind-eu"),
    topology_files=topology_files,
    on_missing_model="warn",
)

premise_region_map.head()

Geomatcher: Used 'GB' for 'UK'
Geomatcher: Used 'GB' for 'UK'
Geomatcher: Used 'GB' for 'UK'
Geomatcher: Used 'GB' for 'UK'


,remind,image,remind-eu
country,,,
AL,NEU,CEU,NES
AT,EUR,WEU,EWN
BA,NEU,CEU,NES
BE,EUR,WEU,EWN
BG,EUR,CEU,ECS


In [51]:
premise_region_map.value_counts(dropna=False)

remind  image  remind-eu
EUR     CEU    ECE          6
               ECS          5
NEU     CEU    NES          5
EUR     WEU    EWN          4
               ESC          3
               ENC          3
               ESW          2
               UKI          2
NEU     WEU    NEN          2
EUR     CEU    ESC          1
        WEU    DEU          1
               FRA          1
Name: count, dtype: int64

### Quick REMIND-EU Check

If `remind-eu-topology.json` is available in `data/tyndp`, the `remind-eu` column should resolve to more detailed European regions instead of `<NA>`.

In [52]:
assert remind_eu_topology.exists(), remind_eu_topology

quick_remind_eu_map = build_premise_region_map(
    ["FR", "DE", "ES", "PT", "IT", "NL", "PL", "UK", "NO"],
    iam_models=("remind-eu",),
    topology_files={"remind-eu": remind_eu_topology},
    on_missing_model="raise",
)

quick_remind_eu_map

,remind-eu
country,
DE,DEU
ES,ESW
FR,FRA
IT,ESC
NL,EWN
NO,NEN
PL,ECE
PT,ESW
UK,UKI


In [53]:
premise_region_map.loc[["FR", "DE", "ES", "PT", "IT", "NL", "PL", "UK", "NO"]]

,remind,image,remind-eu
country,,,
FR,EUR,WEU,FRA
DE,EUR,WEU,DEU
ES,EUR,WEU,ESW
PT,EUR,WEU,ESW
IT,EUR,WEU,ESC
NL,EUR,WEU,EWN
PL,EUR,CEU,ECE
UK,EUR,WEU,UKI
NO,NEU,WEU,NEN


## Map technologies to `premise` activities

Use the TYNDP/electricity concordance matrix to aggregate ENTSO-E technologies into premise activities, then aggregate source countries into premise/IAM regions. Both mappings use xarray matrix multiplication to avoid materializing a large sparse broadcasted array.

In [38]:
premise_consumption_mix_mapper = PremiseConsumptionMixMapper(
    technology_mapping=technology_mapping,
    iam_model="remind-eu",
    topology_files=topology_files,
    on_missing_model="warn",
)
premise_activity_mix_xr = premise_consumption_mix_mapper.map_technologies(
    consumption_mix_xr,
    check=True,
)

premise_activity_mix_xr

<xarray.DataArray 'consumption_mix' (time: 8736, consumer_country: 35,
                                     source_country: 35, premise_activity: 38)> Size: 3GB
array([[[[0.00000000e+00, 0.00000000e+00, 0.00000000e+00, ...,
          0.00000000e+00, 0.00000000e+00, 0.00000000e+00],
         [0.00000000e+00, 0.00000000e+00, 0.00000000e+00, ...,
          0.00000000e+00, 0.00000000e+00, 0.00000000e+00],
         [0.00000000e+00, 0.00000000e+00, 0.00000000e+00, ...,
          0.00000000e+00, 0.00000000e+00, 0.00000000e+00],
         ...,
         [0.00000000e+00, 0.00000000e+00, 0.00000000e+00, ...,
          0.00000000e+00, 0.00000000e+00, 0.00000000e+00],
         [0.00000000e+00, 0.00000000e+00, 0.00000000e+00, ...,
          0.00000000e+00, 0.00000000e+00, 0.00000000e+00],
         [0.00000000e+00, 0.00000000e+00, 0.00000000e+00, ...,
          0.00000000e+00, 0.00000000e+00, 0.00000000e+00]],

        [[0.00000000e+00, 0.00000000e+00, 0.00000000e+00, ...,
          0.00000000e+00, 0.00000000e+00, 0.00000000e+00],
         [4.33001251e-03, 0.00000000e+00, 1.00056066e-02, ...,
          0.00000000e+00, 0.00000000e+00, 4.33001251e-03],
         [0.00000000e+00, 0.00000000e+00, 0.00000000e+00, ...,
          0.00000000e+00, 0.00000000e+00, 0.00000000e+00],
...
          0.00000000e+00, 0.00000000e+00, 0.00000000e+00],
         [1.75414984e-02, 0.00000000e+00, 0.00000000e+00, ...,
          0.00000000e+00, 0.00000000e+00, 1.75414984e-02],
         [5.37308181e-06, 0.00000000e+00, 0.00000000e+00, ...,
          0.00000000e+00, 0.00000000e+00, 5.37308181e-06]],

        [[0.00000000e+00, 0.00000000e+00, 0.00000000e+00, ...,
          0.00000000e+00, 0.00000000e+00, 0.00000000e+00],
         [0.00000000e+00, 0.00000000e+00, 0.00000000e+00, ...,
          0.00000000e+00, 0.00000000e+00, 0.00000000e+00],
         [0.00000000e+00, 0.00000000e+00, 0.00000000e+00, ...,
          0.00000000e+00, 0.00000000e+00, 0.00000000e+00],
         ...,
         [0.00000000e+00, 0.00000000e+00, 0.00000000e+00, ...,
          0.00000000e+00, 0.00000000e+00, 0.00000000e+00],
         [0.00000000e+00, 0.00000000e+00, 0.00000000e+00, ...,
          0.00000000e+00, 0.00000000e+00, 0.00000000e+00],
         [1.39074855e-02, 0.00000000e+00, 0.00000000e+00, ...,
          0.00000000e+00, 0.00000000e+00, 1.39074855e-02]]]],
      shape=(8736, 35, 35, 38))
Coordinates:
  * time              (time) datetime64[ns] 70kB 2050-01-01 ... 2050-12-30T23...
  * consumer_country  (consumer_country) object 280B 'AL' 'AT' ... 'SK' 'UK'
  * source_country    (source_country) object 280B 'AL' 'AT' 'BA' ... 'SK' 'UK'
  * premise_activity  (premise_activity) object 304B 'electricity production,...

In [54]:
premise_region_mix_xr = premise_consumption_mix_mapper.map_regions(
    premise_activity_mix_xr,
    premise_region_map=premise_region_map,
    check=True,
)

premise_region_mix_xr

<xarray.DataArray 'consumption_mix' (time: 8736, consumer_country: 35,
                                     premise_activity: 38, premise_region: 11)> Size: 1GB
array([[[[1.39827159e-04, 0.00000000e+00, 6.05071160e-05, ...,
          0.00000000e+00, 0.00000000e+00, 0.00000000e+00],
         [0.00000000e+00, 0.00000000e+00, 0.00000000e+00, ...,
          0.00000000e+00, 0.00000000e+00, 0.00000000e+00],
         [0.00000000e+00, 0.00000000e+00, 0.00000000e+00, ...,
          0.00000000e+00, 0.00000000e+00, 0.00000000e+00],
         ...,
         [0.00000000e+00, 0.00000000e+00, 0.00000000e+00, ...,
          0.00000000e+00, 0.00000000e+00, 0.00000000e+00],
         [0.00000000e+00, 0.00000000e+00, 0.00000000e+00, ...,
          0.00000000e+00, 0.00000000e+00, 0.00000000e+00],
         [1.39827159e-04, 0.00000000e+00, 6.05071160e-05, ...,
          0.00000000e+00, 0.00000000e+00, 0.00000000e+00]],

        [[4.59010130e-05, 4.33001251e-03, 8.24172367e-04, ...,
          0.00000000e+00, 0.00000000e+00, 0.00000000e+00],
         [0.00000000e+00, 0.00000000e+00, 0.00000000e+00, ...,
          0.00000000e+00, 0.00000000e+00, 0.00000000e+00],
         [0.00000000e+00, 1.00056066e-02, 3.60549733e-03, ...,
          0.00000000e+00, 0.00000000e+00, 0.00000000e+00],
...
          0.00000000e+00, 0.00000000e+00, 0.00000000e+00],
         [0.00000000e+00, 0.00000000e+00, 0.00000000e+00, ...,
          0.00000000e+00, 0.00000000e+00, 0.00000000e+00],
         [0.00000000e+00, 0.00000000e+00, 0.00000000e+00, ...,
          0.00000000e+00, 0.00000000e+00, 5.37599785e-06]],

        [[0.00000000e+00, 0.00000000e+00, 0.00000000e+00, ...,
          0.00000000e+00, 0.00000000e+00, 1.39150333e-02],
         [0.00000000e+00, 0.00000000e+00, 0.00000000e+00, ...,
          0.00000000e+00, 0.00000000e+00, 0.00000000e+00],
         [0.00000000e+00, 0.00000000e+00, 0.00000000e+00, ...,
          0.00000000e+00, 0.00000000e+00, 0.00000000e+00],
         ...,
         [0.00000000e+00, 0.00000000e+00, 0.00000000e+00, ...,
          0.00000000e+00, 0.00000000e+00, 0.00000000e+00],
         [0.00000000e+00, 0.00000000e+00, 0.00000000e+00, ...,
          0.00000000e+00, 0.00000000e+00, 0.00000000e+00],
         [0.00000000e+00, 0.00000000e+00, 0.00000000e+00, ...,
          0.00000000e+00, 0.00000000e+00, 1.39150333e-02]]]],
      shape=(8736, 35, 38, 11))
Coordinates:
  * time              (time) datetime64[ns] 70kB 2050-01-01 ... 2050-12-30T23...
  * consumer_country  (consumer_country) object 280B 'AL' 'AT' ... 'SK' 'UK'
  * premise_activity  (premise_activity) object 304B 'electricity production,...
  * premise_region    (premise_region) object 88B 'NES' 'EWN' ... 'FRA' 'UKI'
Attributes:
    iam_model:  remind-eu

In [55]:
premise_region_mix = (
    premise_region_mix_xr
    .to_dataframe(name="share")
    .reset_index()
)

In [56]:
premise_region_mix.loc[premise_region_mix["consumer_country"].eq("FR")].groupby(
    ["premise_region", "premise_activity"],
    dropna=False,
)["share"].sum().nlargest(30) / 8736

premise_region  premise_activity                                                                                    
FRA             electricity production, photovoltaic, commercial                                                        0.126635
                electricity production, photovoltaic, residential                                                       0.126635
                electricity production, wind, <1MW turbine, onshore                                                     0.116476
                electricity production, wind, >3MW turbine, onshore                                                     0.116476
                electricity production, nuclear, boiling water reactor                                                  0.103897
                electricity production, nuclear, pressure water reactor                                                 0.103897
                electricity production, wind, 1-3MW turbine, offshore                                        